# Exponent- en mantissatruncatie met vaste schaal ×32 / A32 — vijf-epochpilot

Deze proef vergelijkt een **vaste ×32-correctie** met de bestaande ×64-training. E4M3 wordt uit de actuele FP32-gewichten bepaald. E2M3 behoudt de hoogste twee exponentbits van die bron en gebruikt **E8M0-broncode +5**. E2M1 ontstaat vervolgens door twee mantissabits van E2M3 te verwijderen en behoudt dezelfde ×32-schaal. De drie modi delen één E4M3-bron per laag en minibatch.

Activaties blijven **A32**; de gedeelde updatemaster en Adam-momenten blijven **FP32**. De schaalcorrectie wordt niet geleerd. De bronschaal wordt bij volgende minibatches opnieuw uit de actuele gewichten bepaald. Broncode +5 wordt in een breder integer-type berekend; ongeldige E8M0-schalen worden afgewezen zonder clipping of wraparound.

**De twee hoofdgrafieken tonen epoch 5 en het gezamenlijke checkpoint gekozen door de hoogste E4M3-accuracy binnen deze pilot.** Bij een vroege stop wordt de laatste afgeronde epoch duidelijk aangeduid. Epoch 100 wordt in deze proef niet gemeten of voorspeld.

Dit notebook leest uitsluitend opgeslagen JSON/CSV. Het laadt geen dataset, model, checkpoint of optimizer en start geen training.


In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # De runkopie bevat hier het exacte resultaatpad.
SAVE_FIGURES = False  # True schrijft naar een nieuwe analysis_<tijdstip>-map.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

# Gebruik de helper uit déze run, ook als een eerdere notebook dezelfde modulenaam gebruikte.
helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
helper_name = '_mx_triple_scale32_pilot5_analysis_helper'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
if spec is None or spec.loader is None:
    raise FileNotFoundError(f'Analysehelper ontbreekt: {helper_path}')
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Epoch 5 en beste checkpoint gekozen door E4M3

Het beste checkpoint is het **eerste strikte maximum van de E4M3-testaccuracy binnen de pilot**. Een gelijke score vervangt het eerdere checkpoint niet. E2M3 en E2M1 worden op precies dezelfde epoch gerapporteerd; hun eigen hoogste scores worden niet samengevoegd met een andere E4M3-epoch.

De officiële testset wordt volgens het bestaande protocol voor checkpointselectie gebruikt. De beste score is daardoor geen onafhankelijke testschatting na selectie. Het vaste eindpunt epoch 5 staat er apart naast. Dit is een vroege trainingsproef, geen bewijs welke schaalfactor na honderd epochs het beste is.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'Harde test-CE': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


## Vergelijking met vaste schaal ×64 op dezelfde vroege epochs

De meegeleverde ×64-referentie bevat de eerder uitgevoerde honderd-epochtraining, maar de onderstaande vergelijking gebruikt **uitsluitend epochs 1 tot en met 5**. De curves stoppen bij de laatste beschikbare ×32-epoch. De epoch-5-vergelijking verschijnt pas wanneer de ×32-pilot epoch 5 heeft bereikt.

Voor de beste-checkpointvergelijking wordt de hoogste E4M3-accuracy per training opnieuw bepaald **binnen de gezamenlijke vroege epochrange**. Alle drie modi horen telkens bij dat ene checkpoint. De globale beste epoch van de honderd-epochreferentie wordt niet gebruikt. De initialisatie, instellingen en referentiebestanden worden gecontroleerd voordat een vergelijking verschijnt.


In [ ]:
reference = ns.load_scale64_reference(run)
display(Markdown(ns.comparison_markdown(run, reference)))
display(ns.comparison_table(run, reference).round(6))
figures['scale_comparison_curves'] = ns.plot_scale_comparison_curves(run, reference)
display(figures['scale_comparison_curves'])
plt.close(figures['scale_comparison_curves'])
figures['scale_comparison_checkpoints'] = ns.plot_scale_comparison_checkpoints(run, reference)
display(figures['scale_comparison_checkpoints'])
plt.close(figures['scale_comparison_checkpoints'])


## Train- en testaccuracy per modus

De drie modi worden gezamenlijk vanaf een nieuwe initialisatie getraind. Trainaccuracy wordt tijdens de updates verzameld. Testaccuracy wordt na iedere epoch gemeten met dezelfde ×32-truncatieroute, bevroren evaluatiestaat en volledige testset. De drie modusgebonden BatchNorm-banken leren tijdens de training normaal mee.


In [ ]:
figures['accuracy_curves'] = ns.plot_learning(run)
display(figures['accuracy_curves'])
plt.close(figures['accuracy_curves'])


## Losses: trainingsdoelen onderscheiden van harde test-CE

De trainpass van E4M3 gebruikt harde CE van labels. Daarna leert E2M3 met soft-target CE van de **detached E4M3-softmax**; E2M1 leert met soft-target CE van de **detached E2M3-softmax**. Dit is recursieve distillatie met T=1. De drie gradients worden gesommeerd zonder deling door drie, waarna één Adam-update volgt; er is geen extra FP32-teacherforward.

De drie trainobjectieven staan in afzonderlijke panelen omdat hun targets verschillen. Soft CE is geen harde label-CE en wordt hier ook niet als KL-divergentie aangeduid. **Testloss is voor alle drie modi harde CE van dezelfde labels** en staat daarom samen in één vergelijkbaar paneel.


In [ ]:
if not run.metrics.empty:
    objective_columns = [key for key in ('epoch', 'phase', 'weight_format', 'objective_type', 'objective_loss') if key in run.metrics]
    display(run.metrics[objective_columns].round({'objective_loss': 8}))
figures['objective_losses'] = ns.plot_objectives(run)
display(figures['objective_losses'])
plt.close(figures['objective_losses'])


## Blijft het netwerk verschillende cijfers voorspellen?

De eerdere post-hocproeven lieten bij sommige routes sterk vertekende of constante voorspellingen zien. Daarom wordt hier per testepoch en per modus bijgehouden hoe de voorspellingen over de tien cijfers verdeeld zijn. Een constante voorspelling is direct zichtbaar als **één gebruikte klasse en 100% concentratie**. De tabel toont ook hoe vaak het dominante cijfer werkelijk in de testset voorkomt.

Dit zijn geaggregeerde tellingen; deze notebook doet geen nieuwe inferentie en leidt er geen per-beeldvergelijking uit af.


In [ ]:
display(ns.prediction_table(run, selected_only=True))
figures['prediction_diversity'] = ns.plot_prediction_diversity(run)
display(figures['prediction_diversity'])
plt.close(figures['prediction_diversity'])


In [ ]:
display(Markdown('## Gemeten trainingsduur en voorlopige projectie naar honderd epochs'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nDe opgeslagen pieken betreffen de PyTorch CUDA-allocator. Zij meten niet alle GPU-processen of compacte native MX-uitvoering. Ontbrekende waarden blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabellen rapporteren bestaande controles uit de run. Het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates\n\nDe mastergewichten zijn dezelfde gedeelde tensors. De herhaalde masterkolommen per modus mogen **niet bij elkaar worden opgeteld**; code- en schaalwijzigingen behoren wel bij het betreffende formaat.'))
display(ns.weight_update_table(run))
display(Markdown('### Eén opgeslagen E4M3-bron en twee afgeleide exports per geselecteerd checkpoint'))
display(ns.export_table(run))


## Interpretatie en grenzen

De gerichte wijziging tegenover ×64 is **offset +5 in plaats van +6** voor beide lagere precisies. De mantissatruncatie behoudt de E2M3-schaal. Er is geen directe lagere kwantisatie, geen geleerde schaal en geen extra FP32-teacherpass. De identity-STE geldt voor de volledige codebewerking naar G(W), zonder factor32 in de backward; de oorspronkelijke G(W)-afgeleide blijft behouden.

Een lage accuracy of constante voorspelling blijft een onderzoeksuitkomst wanneer de technische controles slagen. Een dalende soft-target trainingsloss bewijst op zichzelf geen betere classificatie. Bekijk daarom testaccuracy, harde test-CE en voorspellingstellingen samen.

De vergelijking met ×64 gebruikt een bestaande afzonderlijke training met dezelfde oorspronkelijke initialisatie en instellingen, afgezien van de vaste correctie. De originele cuDNN-backend is niet bitexact gegarandeerd. Deze proef heeft één seed en vijf epochs; een klein verschil bewijst geen algemene superioriteit of optimale schaalfactor. De beste E4M3-score selecteert op de officiële testset en is geen onafhankelijke testschatting.

De tijdprojectie voor honderd epochs is uitsluitend een looptijdschatting. Er wordt geen accuracy voor epoch 100 berekend. De geheugenfiguur betreft de CUDA-allocator van de FP32-emulator en meet geen native MX-kernel, hardwareversnelling of energiewinst.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')
print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [str(path.relative_to(run.path)) if path.is_relative_to(run.path) else str(path) for path in run.input_files]}))
